### Make imports

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import least_squares


### Load data

In [2]:
df = pd.read_csv(r"xy_data.csv")
x_data = df["x"].values
y_data = df["y"].values


### Known bounds

In [3]:
theta_bounds = (0, np.radians(50))
M_bounds     = (-0.05, 0.05)
X_bounds     = (0, 100)
t_bounds     = (6, 60)

LOWER = [theta_bounds[0], M_bounds[0], X_bounds[0]]
UPPER = [theta_bounds[1], M_bounds[1], X_bounds[1]]


### Residual function 

In [4]:
def residuals(params, x, y):
    theta, M, X = params
    u = (x - X) * np.cos(theta) + (y - 42) * np.sin(theta)
    v = -(x - X) * np.sin(theta) + (y - 42) * np.cos(theta)
    pred_v = np.exp(M * np.abs(u)) * np.sin(0.3 * u)
    return v - pred_v


### Global convergence verification

In [5]:
N_STARTS = 50
rng = np.random.default_rng(0)

all_results = []
for _ in range(N_STARTS):
    x0 = [rng.uniform(*theta_bounds), rng.uniform(*M_bounds), rng.uniform(*X_bounds)]
    sol = least_squares(residuals, x0=x0, args=(x_data, y_data), bounds=(LOWER, UPPER))
    all_results.append((sol.x, sol.cost))

# Sort by cost, best first
all_results.sort(key=lambda r: r[1])
best_params, best_cost = all_results[0]

# convergence to same answer
tol = 1e-6
n_matching = sum(
    np.allclose(r[0], best_params, atol=tol) for r in all_results
)


### Answer

In [6]:
theta, M, X = best_params
theta_deg = np.degrees(theta)

print("Global convergence verification")
print(f"Best cost:            {best_cost}")
print(f"Runs converging to best solution: {n_matching}/{N_STARTS}")
print()

print("Fitted parameters/Answers")
print(f"theta = {theta_deg} deg  ({theta} rad)")
print(f"M     = {M}")
print(f"X     = {X}")
print()

# recovered t range should match known t range (6 to 60)
t_vals = (x_data - X) * np.cos(theta) + (y_data - 42) * np.sin(theta)
print("t-Range Verification")
print(f"Known t range:     {t_bounds}")
print(f"Recovered t range: ({t_vals.min()}, {t_vals.max()})")

Global convergence verification
Best cost:            9.114989679557451e-09
Runs converging to best solution: 38/50

Fitted parameters/Answers
theta = 29.99997293215849 deg  (0.5235983031753431 rad)
M     = 0.029999996873053714
X     = 54.99999821279995

t-Range Verification
Known t range:     (6, 60)
Recovered t range: (6.049405472717945, 59.995170702349114)
